# Preparación reproducible de datos · E3

La preparación estudia la calidad de identidades públicas de perros y gatos antes de ajustar el modelo. El pipeline adquiere un piloto acotado, verifica el detector congelado, filtra fotos ambiguas, elimina duplicados y divide por individuo. Produce un dataset versionado y portable, con conteos y motivos de cada exclusión.

YOLO reduce ambigüedad de escena; una sola detección no verifica identidad ni sesiones distintas. El protocolo exploratorio conserva este límite en sus reportes. Esta etapa no entrena, no registra modelos y no descarga el benchmark privado de PetTrace.


In [ ]:
%pip install -q --only-binary=:all: "numpy>=2.1,<2.3" "pillow>=11,<13" "pyyaml>=6,<7" "onnxruntime==1.20.1" "mlflow-skinny==3.16.1" "requests>=2.32,<3"


In [ ]:
import importlib.util
import os
# Colab commonly already has torch; import it first when present.
try:
    import torch
except ImportError:
    pass
PACKAGE_SOURCE = os.environ.get("PETTRACE_PACKAGE_SOURCE", "git+https://github.com/alessandro54/pettrace-ml.git@main")
try:
    has_preparation = importlib.util.find_spec("pettrace.e3_prepare") is not None
except ModuleNotFoundError:
    has_preparation = False
if not has_preparation or os.environ.get("PETTRACE_PACKAGE_SOURCE"):
    get_ipython().run_line_magic("pip", f'install -q --upgrade --force-reinstall --no-deps "{PACKAGE_SOURCE}"')
importlib.invalidate_caches()
from pettrace.e3_prepare import PrepareConfig, prepare, acquire_detector


In [ ]:
import json
from pathlib import Path
from IPython.display import Image, display

DATA_ROOT = Path(os.environ.get("E3_DATA_ROOT", "datasets/public/e3-body")).resolve()
OUTPUT_ROOT = DATA_ROOT / "prepared"
DETECTOR_DIR = os.environ.get("E3_DETECTOR_DIR")
BASE_MODEL = "models:/pettrace-embedder/2"
SOURCE_ROOTS = {name: value for name, value in {
    "dogs-world": os.environ.get("E3_DOGS_WORLD_ROOT"),
    "lcw": os.environ.get("E3_LCW_ROOT"),
}.items() if value}
IDENTITIES_PER_SPECIES = 300
PHOTOS_PER_IDENTITY = 10
SEED = 42
ALLOW_BOUNDED_DOWNLOAD = True


## Acceso al detector

Se reutilizan únicamente `detector.onnx` y su especificación de la versión E2, con hash fijado. Los accesos MLflow son de lectura. `E3_DETECTOR_DIR` puede señalar esos archivos locales y evita la descarga. Esta etapa no solicita `HF_TOKEN`.


In [ ]:
# Public pilot preparation needs no private credentials.
# Request only the accesses needed by each stage; never print credential values.
def configure_model_access(names=("MLFLOW_TRACKING_URI", "MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD")):
    # Credentials stay in environment variables / Secrets / hidden prompts, never in notebook source.
    import getpass
    import os
    import warnings

    try:
        from google.colab import userdata
    except ImportError:
        userdata = None

    for name in names:
        if not os.environ.get(name) and userdata is not None:
            try:
                os.environ[name] = userdata.get(name)
            except Exception:
                pass  # VS Code may not expose Secrets; use a hidden prompt below.
        if not os.environ.get(name):
            if name == "MLFLOW_TRACKING_URI":
                value = "https://ml.pettrace.app"
            else:
                # Fail before reading input if the kernel cannot hide the entered value.
                with warnings.catch_warnings():
                    warnings.simplefilter("error", getpass.GetPassWarning)
                    value = getpass.getpass(f"{name}: ")
            if not value.strip():
                raise ValueError(f"Missing {name}.")
            os.environ[name] = value


## Preparación y versión de salida

La adquisición usa rangos HTTP y reutiliza los archivos ya descargados. YOLO admite exactamente un perro o gato de la especie esperada. La limpieza respeta decisiones manuales existentes, excluye identidades ambiguas conocidas, pone en cuarentena conflictos y exige cuatro fotos distintas. Solo se publica localmente una versión terminada con suficientes identidades por split.

Una repetición verifica y reutiliza la misma versión cuando los inputs y parámetros coinciden. Cambios en imágenes, decisiones, código o configuración producen otra huella. Las versiones previas permanecen disponibles.


In [ ]:
if DETECTOR_DIR:
    detector_dir = Path(DETECTOR_DIR).resolve()
else:
    configure_model_access()
    detector_dir = acquire_detector(BASE_MODEL, DATA_ROOT / "detector-models")
config = PrepareConfig(
    data_root=str(DATA_ROOT), detector_dir=str(detector_dir), output_root=str(OUTPUT_ROOT),
    source_roots=SOURCE_ROOTS, identities_per_species=IDENTITIES_PER_SPECIES,
    photos_per_identity=PHOTOS_PER_IDENTITY, seed=SEED, allow_download=ALLOW_BOUNDED_DOWNLOAD)
prepared = prepare(config)
print("Prepared dataset:", prepared.root)
print("Fingerprint:", prepared.fingerprint)
print(json.dumps(prepared.summary, indent=2))


## Muestra visual y trazabilidad

Se muestran cuatro individuos por especie seleccionados con semilla fija después del filtrado. Es una inspección por muestra; las etiquetas pendientes conservan su estado sin verificar. `excluded.csv` registra todos los descartes y `animal-screening.json` conserva cajas, puntuaciones y motivos.


In [ ]:
sample = json.loads((prepared.root / "visual-sample.json").read_text())
for item in sample["sample"]:
    print(item["animal"])
    display(Image(filename=str(prepared.root / item["sheet"])))


## Consumo en entrenamiento

El siguiente paso es `e3_mlp_head_triplet.ipynb`. En el mismo runtime encuentra automáticamente `DATA_ROOT/prepared/latest.json`; también puede fijarse `E3_PREPARED_DATASET` a la carpeta de una versión específica. Para otro runtime se copia la carpeta completa de esa versión, con sus imágenes.

El entrenamiento verifica los hashes y los splits antes de comenzar. Preparar datos nuevamente no inicia entrenamiento; entrenar de nuevo no repite la adquisición ni YOLO.
